# Module 1 · Session 2 — Challenge Solutions

Use this notebook **after attempting the challenge**. The purpose is to compare reasoning and implementation, not merely to obtain outputs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

dates = pd.to_datetime(["2026-06-01", "2026-06-15", "2026-07-01"])
latitudes = [51.50, 51.49, 51.48]
longitudes = [-3.20, -3.19, -3.18]

ndvi_data = np.array([
    [[0.72, 0.68, 0.31], [0.75, 0.70, 0.28], [0.81, 0.74, 0.25]],
    [[0.74, 0.71, 0.35], [0.78, 0.73, 0.30], [0.84, 0.77, 0.27]],
    [[0.70, 0.67, 0.29], [0.73, 0.69, 0.26], [0.79, 0.72, 0.23]]
])

temperature_data = np.array([
    [[18.2, 18.5, 19.1], [17.9, 18.4, 19.3], [17.5, 18.0, 18.8]],
    [[21.1, 21.4, 22.0], [20.8, 21.3, 22.2], [20.4, 20.9, 21.7]],
    [[24.3, 24.7, 25.5], [24.0, 24.5, 25.8], [23.6, 24.2, 25.1]]
])

## Task 1 — Solution

In [ ]:
ds = xr.Dataset(
    {
        "ndvi": (["time", "latitude", "longitude"], ndvi_data),
        "temperature": (["time", "latitude", "longitude"], temperature_data),
    },
    coords={
        "time": dates,
        "latitude": latitudes,
        "longitude": longitudes,
    },
    attrs={"description": "NDVI and temperature observations over time and space"},
)

ds["ndvi"].attrs = {
    "long_name": "Normalized Difference Vegetation Index",
    "units": "1",
}
ds["temperature"].attrs = {
    "long_name": "Land Surface Temperature",
    "units": "degC",
}

ds

## Task 2 — Solution

In [ ]:
point = ds.sel(
    time="2026-07-01",
    latitude=51.48,
    longitude=-3.19
)
point

At this coordinate on 1 July 2026, NDVI is 0.72 and land-surface temperature is 24.2 degC.

## Task 3 — Solution

In [ ]:
series = ds["ndvi"].sel(latitude=51.50, longitude=-3.20)
change = series.diff(dim="time")

print(series)
print(change)

NDVI increases from 0.72 to 0.74 (+0.02), then decreases from 0.74 to 0.70 (-0.04).

## Task 4 — Solution

In [ ]:
mean_ndvi = ds["ndvi"].mean(dim=["latitude", "longitude"])
mean_temperature = ds["temperature"].mean(dim=["latitude", "longitude"])

print(mean_ndvi)
print(mean_temperature)

## Task 5 — Solution

In [ ]:
quality = xr.DataArray(
    [[True, True, False],
     [True, False, False],
     [True, True, True]],
    dims=["latitude", "longitude"],
    coords={"latitude": latitudes, "longitude": longitudes}
)

first_ndvi = ds["ndvi"].sel(time="2026-06-01")
masked = first_ndvi.where(quality)

print("Original mean:", float(first_ndvi.mean()))
print("Masked mean:", float(masked.mean()))
print(masked)

Masking can raise or lower a statistic because the direction depends on the values that the quality criteria exclude.

## Task 6 — Solution

In [ ]:
missing_ds = ds.copy(deep=True)
missing_ds["ndvi"][1, 1, 2] = np.nan

possible = missing_ds["ndvi"].size
valid = int(missing_ds["ndvi"].count())
completeness = valid / possible * 100

print("Possible:", possible)
print("Non-missing:", valid)
print("Completeness:", round(completeness, 2), "%")

## Task 7 — Solution

In [ ]:
output_dir = Path("../data")
output_dir.mkdir(exist_ok=True)

challenge_file = output_dir / "session2_challenge.nc"
ds.to_netcdf(challenge_file)

reloaded_ds = xr.open_dataset(challenge_file)
reloaded_ds

## Task 8 — Solution

In [ ]:
mean_temp = reloaded_ds["temperature"].mean(dim=["latitude", "longitude"])
mean_ndvi = reloaded_ds["ndvi"].mean(dim=["latitude", "longitude"])

highest_temp_date = mean_temp.idxmax(dim="time")
highest_temp = mean_temp.max()

highest_ndvi_date = mean_ndvi.idxmax(dim="time")

point_series = reloaded_ds["ndvi"].sel(latitude=51.49, longitude=-3.18)
net_change = point_series.isel(time=-1) - point_series.isel(time=0)

print("Highest mean temperature date:", highest_temp_date.values)
print("Highest mean temperature:", round(float(highest_temp.values), 2), "degC")
print("Highest mean NDVI date:", highest_ndvi_date.values)
print("Net NDVI change at (51.49, -3.18):", round(float(net_change.values), 2))

## Expected conclusions

- Highest spatial mean temperature: **1 July 2026**, approximately **24.63 degC**.
- Highest spatial mean NDVI: **15 June 2026**.
- At latitude 51.49, longitude -3.18, NDVI changes from 0.28 to 0.26, giving a net change of **-0.02**.

A real EO product would also need stronger geospatial metadata (for example CRS/georeferencing conventions), documented provenance, quality information, and appropriate data validation.